!pip install pymysql sqlalchemy pandas

## Connessione al Database e Importazione Dati
In questa prima fase vado a creare un "ponte" tra l'ambiente Python e il mio database MySQL. 
Invece di esportare pesanti file CSV, ho deciso di interrogare direttamente il database usando la libreria `SQLAlchemy`. Questo mi permette di estrarre solo le due tabelle aggregate che ho creato in precedenza (Logistica e Marketing), ottimizzando l'uso della memoria RAM del mio computer.

In [1]:
import pandas as pd
from sqlalchemy import create_engine

# Imposto le credenziali
user = 'root'
password = 'Candreva87'
host = 'localhost'
database = 'hm_retail_db'

# Creo il motore di connessione
print("Connessione a MySQL in corso...")
engine = create_engine(f'mysql+pymysql://{user}:{password}@{host}/{database}')

# Importo le tabelle aggregate e il catalogo
print("Estrazione della tabella Logistica (daily_sales)...")
df_sales = pd.read_sql("SELECT * FROM daily_sales", con=engine)

print("Estrazione della tabella Marketing (customer_insights)...")
df_customers = pd.read_sql("SELECT * FROM customer_insights", con=engine)

print("Estrazione del Catalogo Prodotti (articles)...")
df_articles = pd.read_sql("SELECT * FROM articles", con=engine)

print("Operazione completata! Tutti i dati necessari sono in memoria.")

Connessione a MySQL in corso...
Estrazione della tabella Logistica (daily_sales)...
Estrazione della tabella Marketing (customer_insights)...
Estrazione del Catalogo Prodotti (articles)...
Operazione completata! Tutti i dati necessari sono in memoria.


##  Pulizia Dati Generale
In questa seconda fase mi occuperò del Data Cleaning totale,ad esempio riempiremo le età mancanti con una mediana,elimineremo articoli fantasma senza nomi associati o vendite che sono negative (impossibile),insomma mi occuperò di portare i dati a PowerBI impeccabili per la nostra 3 fase.

In [2]:
import numpy as np

# 1. Converto in modo forzato tutti i valori non numerici in 'NaN' (Not a Number)
df_customers['age'] = pd.to_numeric(df_customers['age'], errors='coerce')

# 2. Ora conto di nuovo i mancanti
missing_ages = df_customers['age'].isnull().sum()
print(f"Ora vedo correttamente {missing_ages} clienti senza età.")



Ora vedo correttamente 12038 clienti senza età.


In [3]:
# 3. Ora applico la mediana
if missing_ages > 0:
    median_age = df_customers['age'].median()
    df_customers['age'] = df_customers['age'].fillna(median_age)
    print(f"✅ Ho riempito i {missing_ages} buchi con l'età media: {int(median_age)} anni.")

✅ Ho riempito i 12038 buchi con l'età media: 32 anni.


In [4]:
# --- 0. PULIZIA DEI NOMI DELLE COLONNE (Risoluzione del KeyError) ---
# Pulisco i nomi delle colonne da eventuali spazi o caratteri strani importati dal Wizard
df_sales.columns = df_sales.columns.str.replace(r'[^a-zA-Z0-9_]', '', regex=True).str.lower()
df_articles.columns = df_articles.columns.str.replace(r'[^a-zA-Z0-9_]', '', regex=True).str.lower()

print("Nomi colonne df_sales ripuliti:", df_sales.columns.tolist())
print("Nomi colonne df_articles ripuliti:", df_articles.columns.tolist())

# --- 1. PULIZIA TESTI (Standardizzazione) ---
df_articles['prod_name'] = df_articles['prod_name'].str.strip().str.lower()
df_articles['colour_group_name'] = df_articles['colour_group_name'].str.strip().str.lower()

# --- 2. GESTIONE VALORI MANCANTI NEL CATALOGO ---
df_articles['prod_name'] = df_articles['prod_name'].fillna('n/a')
df_articles['colour_group_name'] = df_articles['colour_group_name'].fillna('n/a')

# --- 3. CONVERSIONE DATE ---
df_sales['t_dat'] = pd.to_datetime(df_sales['t_dat'])

# --- 4. RIMOZIONE OUTLIER (Prezzi assurdi) ---
df_sales = df_sales[df_sales['total_revenue'] > 0]


# --- 5. VALIDAZIONE INCROCIATA (Eliminazione ID fantasma) ---
df_sales = df_sales[df_sales['article_id'].isin(df_articles['article_id'])]

print("\n✅ Pulizia completa: L'errore KeyError è stato risolto e i dati sono standardizzati.")

Nomi colonne df_sales ripuliti: ['t_dat', 'article_id', 'total_pieces', 'total_revenue']
Nomi colonne df_articles ripuliti: ['article_id', 'product_code', 'prod_name', 'product_type_no', 'product_type_name', 'product_group_name', 'graphical_appearance_no', 'graphical_appearance_name', 'colour_group_code', 'colour_group_name', 'perceived_colour_value_id', 'perceived_colour_value_name', 'perceived_colour_master_id', 'perceived_colour_master_name', 'department_no', 'department_name', 'index_code', 'index_name', 'index_group_no', 'index_group_name', 'section_no', 'section_name', 'garment_group_no', 'garment_group_name', 'detail_desc']

✅ Pulizia completa: L'errore KeyError è stato risolto e i dati sono standardizzati.


##  Inventory Analytics: Calcolo del Reorder Point (ROP) e Simulazione Stock
Essendo il dataset originale basato esclusivamente sui dati transazionali (vendite), ho sviluppato un algoritmo per simulare la **Giacenza Attuale** del magazzino in modo proporzionale ai volumi storici di vendita di ciascun articolo, garantendo un ambiente di test realistico.

Successivamente, ho applicato i modelli matematici standard della Supply Chain per automatizzare il riordino merci. Ho calcolato la media di vendita giornaliera per ogni capo e definito:
* Un **Lead Time** di 7 giorni (tempo di approvvigionamento standard).
* Un **Safety Stock** (scorta di sicurezza) per coprire eventuali picchi di domanda o ritardi.
* Il **Reorder Point (ROP)**, ovvero la soglia matematica al di sotto della quale il sistema deve generare un alert automatico per l'ufficio acquisti.

In [5]:
import numpy as np

# 1. Raggruppo le vendite per articolo
# Calcolo i pezzi totali venduti e i giorni in cui l'articolo è stato effettivamente venduto
df_inventory = df_sales.groupby('article_id').agg(
    total_sold=('total_pieces', 'sum'),
    days_active=('t_dat', 'nunique')
).reset_index()

# Calcolo la media di vendita giornaliera (arrotondata per eccesso)
df_inventory['avg_daily_sales'] = np.ceil(df_inventory['total_sold'] / df_inventory['days_active'])

# 2. Simulazione della "Giacenza Attuale" (Current Stock)
# Genero uno stock che copra casualmente tra i 15 e i 30 giorni di vendite per simulare un magazzino reale
np.random.seed(42) # Questo fissa i numeri casuali così avrò sempre gli stessi risultati
df_inventory['current_stock'] = (df_inventory['avg_daily_sales'] * np.random.randint(15, 30, size=len(df_inventory))).astype(int)

# 3. Calcoli per la Supply Chain (ROP & Safety Stock)
lead_time_days = 7  # Tempo di consegna fornitore
safety_stock_days = 3 # Giorni di cuscinetto

# Scorta di sicurezza = vendite medie giornaliere * giorni di cuscinetto
df_inventory['safety_stock'] = df_inventory['avg_daily_sales'] * safety_stock_days

# Reorder Point = (Vendite medie * Tempo consegna) + Scorta di sicurezza
df_inventory['reorder_point'] = (df_inventory['avg_daily_sales'] * lead_time_days) + df_inventory['safety_stock']

# 4. Generazione dell'Allarme di Riordino
# Se la giacenza scende sotto il ROP, scatta l'allarme!
df_inventory['restock_alarm'] = np.where(df_inventory['current_stock'] <= df_inventory['reorder_point'], '🚨 ORDINE URGENTE', '✅ STOCK OK')

print("✅ Calcolo ROP e Simulazione Magazzino completati!")
display(df_inventory.head())

✅ Calcolo ROP e Simulazione Magazzino completati!


,article_id,total_sold,days_active,avg_daily_sales,current_stock,safety_stock,reorder_point,restock_alarm


##  Markdown Strategy: Identificazione dell'Overstock
Un'efficiente gestione della Supply Chain non riguarda solo evitare gli stockout (mancanza di merce), ma anche ottimizzare lo spazio in magazzino. Per questo motivo, ho implementato una regola logica per identificare i prodotti in **Overstock** (giacenza in eccesso) che necessitano di un'azione commerciale.

L'algoritmo valuta due condizioni contemporaneamente:
1. **Bassa rotazione:** L'articolo ha una media di vendita giornaliera molto bassa (es. meno di 2 pezzi al giorno).
2. **Giacenza elevata:** Lo stock attuale supera una determinata soglia di allerta (es. più di 25 pezzi in magazzino).

I prodotti che soddisfano entrambe le condizioni vengono automaticamente etichettati come "🔴 DA SCONTARE", fornendo un input diretto al reparto Marketing per attivare promozioni mirate e liberare capitale immobilizzato.

In [6]:
# Definisco le regole matematiche per l'Overstock
# Condizione 1: Il prodotto vende pochissimo (meno di 2 pezzi in media al giorno)
condizione_lento = df_inventory['avg_daily_sales'] <= 2

# Condizione 2: Abbiamo troppi pezzi a prendere polvere in magazzino (più di 25)
condizione_eccesso = df_inventory['current_stock'] > 25

# Applico la regola: se ENTRAMBE le condizioni sono vere (&), scatta lo sconto
df_inventory['markdown_status'] = np.where(condizione_lento & condizione_eccesso, '🔴 DA SCONTARE', '🟢 PREZZO PIENO')

# Mostro i risultati concentrandoci sulle nuove colonne
print("✅ Strategia di Sconto (Markdown) applicata con successo!")
display(df_inventory[['article_id', 'avg_daily_sales', 'current_stock', 'markdown_status']].head(15))

✅ Strategia di Sconto (Markdown) applicata con successo!


,article_id,avg_daily_sales,current_stock,markdown_status


##  Customer Analytics: Segmentazione e Profilazione Marketing
Dopo aver definito lo stato logistico degli articoli, passo all'analisi dei consumatori. 
Il primo step del Customer Profiling consiste nella segmentazione del database clienti: ho trasformato l'età numerica in "Fasce d'età" (Generazionali) per rendere i dati aggregabili e interpretabili a livello di marketing. In parallelo, sfrutterò lo status di fidelizzazione (Club Member) per capire l'affidabilità del target.

In [7]:
# Creo i "tagli" per le età (es. da 15 a 25 anni, da 25 a 35, ecc.)
bins = [15, 25, 35, 50, 100]

# Do un nome commerciale a questi gruppi
labels = ['16-25 (Gen Z)', '26-35 (Millennials)', '36-50 (Gen X)', 'Over 50 (Boomers)']

# Applico il taglio alla  tabella clienti, creando la nuova colonna 'age_group'
df_customers['age_group'] = pd.cut(df_customers['age'], bins=bins, labels=labels)

print("✅ Segmentazione Clienti completata. Ecco le fasce d'età:")
display(df_customers[['customer_id', 'age', 'age_group', 'club_member_status']].head())

✅ Segmentazione Clienti completata. Ecco le fasce d'età:


,customer_id,age,age_group,club_member_status
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,49.0,36-50 (Gen X),ACTIVE
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,25.0,16-25 (Gen Z),ACTIVE
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,24.0,16-25 (Gen Z),ACTIVE
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,54.0,Over 50 (Boomers),ACTIVE
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,52.0,Over 50 (Boomers),ACTIVE


## Cross-Analysis: Allineamento Logistica - Marketing
In questo passaggio collego fisicamente i problemi di magazzino (Supply Chain) con le soluzioni anagrafiche (Marketing). 

Poiché i report di vendita giornalieri aggregati non contengono il dettaglio del singolo acquirente, ho interrogato direttamente il database SQL estraendo il registro grezzo degli scontrini (`transactions`). Filtrando esclusivamente gli articoli in "Overstock" (contrassegnati come "DA SCONTARE"), ho incrociato ogni singola transazione con il database clienti (`customers`). 

Questo mi ha permesso di identificare la "Buyer Persona" esatta per la merce ferma in magazzino. L'output finale fornisce all'ufficio marketing un target demografico (Fascia d'età e Status) estremamente preciso su cui lanciare campagne promozionali mirate, massimizzando il tasso di conversione e svuotando il magazzino in modo efficiente.

In [ ]:
# 1. Pulizia preventiva: sistemo i nomi delle colonne dei clienti per evitare errori nascosti
df_customers.columns = df_customers.columns.str.replace(r'[^a-zA-Z0-9_]', '', regex=True).str.lower()

# 2. Vado su MySQL a prendere gli scontrini (solo ID Cliente e ID Articolo per fare l'incrocio)
print("⏳ Estrazione degli scontrini (transactions) da MySQL in corso...")
df_transactions = pd.read_sql("SELECT customer_id, article_id FROM transactions", con=engine)
print("✅ Scontrini importati con successo!")

# 3. L'INCROCIO CORRETTO
# Isolo i prodotti che ho messo in Overstock ("DA SCONTARE")
overstock_articles = df_inventory[df_inventory['markdown_status'] == '🔴 DA SCONTARE'][['article_id']]

# Incrocio 1: Quali scontrini contengono gli articoli in overstock?
overstock_sales = pd.merge(overstock_articles, df_transactions, on='article_id', how='inner')

# Incrocio 2: Chi sono i clienti (età e status) di quegli scontrini?
overstock_target = pd.merge(overstock_sales, df_customers, on='customer_id', how='inner')

# 4. Calcolo la classifica: quale fascia di età ne compra di più?
target_analysis = overstock_target.groupby(['age_group', 'club_member_status'], observed=True)['article_id'].count().reset_index()
target_analysis.rename(columns={'article_id': 'pezzi_comprati'}, inplace=True)

# Ordino i risultati dal gruppo più promettente a quello meno promettente
target_analysis = target_analysis.sort_values(by='pezzi_comprati', ascending=False)

print("\n🎯 PROFILAZIONE OVERSTOCK COMPLETATA!")
print("Ecco la classifica del target ideale per le promozioni:")
display(target_analysis.head(10))

⏳ Estrazione degli scontrini (transactions) da MySQL in corso...


##  Esportazione dei Dataset per la Data Visualization (BI)
A conclusione della fase di Data Engineering e Analytics, procedo all'esportazione dei dataset definitivi. 
Tutte le tabelle sono state pulite, standardizzate e arricchite con le logiche di business (Inventory e Customer Analytics). L'esportazione avviene in formato CSV per garantire massima compatibilità e leggerezza. 


In [ ]:
print("⏳ Inizio esportazione dei file definitivi per Power BI...")

# 1. Esportiamo il Magazzino (con allarmi ROP e Sconti)
df_inventory.to_csv('hm_inventory_final.csv', index=False)
print("1/5 - Tabella Magazzino salvata!")

# 2. Esportiamo i Clienti (con l'età pulita e le fasce d'età 'age_group')
df_customers.to_csv('hm_customers_final.csv', index=False)
print("2/5 - Tabella Clienti salvata!")

# 3. Esportiamo le Vendite (pulite da outlier e con date convertite)
df_sales.to_csv('hm_sales_final.csv', index=False)
print("3/5 - Tabella Vendite salvata!")

# 4. Esportiamo il Catalogo Articoli (nomi e colori standardizzati in minuscolo)
df_articles.to_csv('hm_articles_final.csv', index=False)
print("4/5 - Catalogo Articoli salvato!")

# 5. Esportiamo la Classifica del Target Marketing per l'Overstock
target_analysis.to_csv('hm_target_overstock.csv', index=False)
print("5/5 - Tabella Target Marketing salvata!")

print("\n🎉 FASE 2 COMPLETATA AL 100%! 🎉")
print("I file CSV sono pronti nella tua cartella di lavoro.")

## Data Validation ed Esportazione Definitiva (Risoluzione Anomalie)
Durante il primo tentativo di join ed esportazione, è emersa un'anomalia tecnica (tabelle risultanti vuote). Il problema è stato diagnosticato come un *Data Type Mismatch* (mancata corrispondenza dei formati) sulle chiavi di unione `article_id`. Il database originale trasmetteva alcuni identificativi come testo con zeri iniziali (es. "0123") e altri come valori numerici con virgola mobile (es. "123.0"). Questo ha causato il fallimento dei filtri di corrispondenza.

Per garantire la totale integrità dei dati, questa pipeline finale esegue una procedura di ripristino e validazione:
* **Re-fetch dei dati:** Estrazione pulita da MySQL per ripristinare i dataframe in memoria.
* **Standardizzazione Assoluta delle Chiavi:** Creazione di una funzione dedicata (`pulisci_id`) che forza il formato stringa, rimuove i decimali fantasma tramite Regex e normalizza gli zeri iniziali.
* **Ricalcolo delle Metriche:** Riapplicazione delle logiche di Inventory Analytics e Marketing Profiling sui dati validati.
* **Esportazione "Golden Source":** Salvataggio finale dei file CSV puliti e popolati, pronti per l'ingestion in Power BI.

In [ ]:
import numpy as np
import pandas as pd

print("⏳ 1/4 - Ricaricamento dei dati intatti da MySQL in corso...")
# Andiamo a riprendere i dati freschi visto che in memoria si erano svuotati
df_sales_raw = pd.read_sql("SELECT * FROM daily_sales", con=engine)
df_articles_raw = pd.read_sql("SELECT * FROM articles", con=engine)

print("⏳ 2/4 - Pulizia colonne e formati in corso...")
# Pulizia nomi colonne e date
df_sales_raw.columns = df_sales_raw.columns.str.replace(r'[^a-zA-Z0-9_]', '', regex=True).str.lower()
df_articles_raw.columns = df_articles_raw.columns.str.replace(r'[^a-zA-Z0-9_]', '', regex=True).str.lower()
df_sales_raw['t_dat'] = pd.to_datetime(df_sales_raw['t_dat'])
df_sales_raw = df_sales_raw[df_sales_raw['total_revenue'] > 0]

# LA CURA D'URTO PER GLI ID
def pulisci_id(colonna):
    return colonna.astype(str).str.replace(r'\.0$', '', regex=True).str.lstrip('0')

df_sales_raw['article_id'] = pulisci_id(df_sales_raw['article_id'])
df_articles_raw['article_id'] = pulisci_id(df_articles_raw['article_id'])
df_transactions['article_id'] = pulisci_id(df_transactions['article_id'])

print("⏳ 3/4 - Ricalcolo Magazzino e Target Marketing...")
# Riapplichiamo il filtro (ORA FUNZIONERÀ PERCHÉ I DATI SONO FRESCHI E PULITI)
df_sales_pulite = df_sales_raw[df_sales_raw['article_id'].isin(df_articles_raw['article_id'])]

# Magazzino
df_inventory = df_sales_pulite.groupby('article_id').agg(
    total_sold=('total_pieces', 'sum'),
    days_active=('t_dat', 'nunique')
).reset_index()

df_inventory['avg_daily_sales'] = np.ceil(df_inventory['total_sold'] / df_inventory['days_active'])
np.random.seed(42)
df_inventory['current_stock'] = (df_inventory['avg_daily_sales'] * np.random.randint(15, 30, size=len(df_inventory))).astype(int)
df_inventory['safety_stock'] = df_inventory['avg_daily_sales'] * 3
df_inventory['reorder_point'] = (df_inventory['avg_daily_sales'] * 7) + df_inventory['safety_stock']
df_inventory['restock_alarm'] = np.where(df_inventory['current_stock'] <= df_inventory['reorder_point'], '🚨 ORDINE URGENTE', '✅ STOCK OK')

condizione_lento = df_inventory['avg_daily_sales'] <=2
condizione_eccesso = df_inventory['current_stock'] > 25
df_inventory['markdown_status'] = np.where(condizione_lento & condizione_eccesso, '🔴 DA SCONTARE', '🟢 PREZZO PIENO')

# Target Marketing
overstock_articles = df_inventory[df_inventory['markdown_status'] == '🔴 DA SCONTARE'][['article_id']]
overstock_sales = pd.merge(overstock_articles, df_transactions, on='article_id', how='inner')
overstock_target = pd.merge(overstock_sales, df_customers, on='customer_id', how='inner')
target_analysis = overstock_target.groupby(['age_group', 'club_member_status'], observed=True)['article_id'].count().reset_index()
target_analysis.rename(columns={'article_id': 'pezzi_comprati'}, inplace=True)
target_analysis = target_analysis.sort_values(by='pezzi_comprati', ascending=False)

print("⏳ 4/4 - Salvataggio file CSV definitivi...")
# SOVRASCRIVIAMO I FILE
df_sales_pulite.to_csv('hm_sales_final.csv', index=False)
df_inventory.to_csv('hm_inventory_final.csv', index=False)
target_analysis.to_csv('hm_target_overstock.csv', index=False)
df_articles_raw.to_csv('hm_articles_final.csv', index=False)

print("\n✅ RISOLTO AL 100%!")
print(f"Vendite recuperate: {len(df_sales_pulite)} righe")
print(f"Magazzino recuperato: {len(df_inventory)} righe")
print(f"Target Marketing recuperato: {len(target_analysis)} righe")
